In [ ]:
import os
import shutil
import random

import cv2
import albumentations as A

In [ ]:
INPUT_IMAGE_DIR = "dataset_no-aug/Raw/valid/Eosinophil"
OUTPUT_IMAGE_DIR = "dataset_aug/Raw/valid/Eosinophil"

MAX_CLASS_REFERENCE = 1150

BG_COLOR_BGR = (196, 208, 188)
BG_COLOR_RGB = BG_COLOR_BGR[::-1]

In [ ]:
# Define Augmentation Pipeline
augment_pipeline = A.Compose([ 
     
    # Image Flipping
    A.HorizontalFlip(p=0.5), 
     
    A.VerticalFlip(p=0.5), 

    # Image Rotation
    A.Rotate( 
        limit=180, 
        border_mode=cv2.BORDER_CONSTANT, 
        fill=BG_COLOR_RGB, 
        p=1.0 
    ), 

    # Image Distortion
    A.OneOf([ 
        A.ElasticTransform( 
            alpha=35, 
            sigma=10, 
            border_mode=cv2.BORDER_CONSTANT, 
            fill=BG_COLOR_RGB, 
            p=1.0 
        ), 

        A.GridDistortion( 
            num_steps=5, 
            distort_limit=0.20, 
            border_mode=cv2.BORDER_CONSTANT, 
            fill=BG_COLOR_RGB, 
            p=1.0 
        ) 
    ], p=0.70), 

    # Affine Transformation
    A.Affine( 
        shear={ 
            "x": (-7, 7), 
            "y": (-7, 7) 
        }, 
        scale=(0.90, 1.05), 
        border_mode=cv2.BORDER_CONSTANT, 
        fill=BG_COLOR_RGB, 
        p=0.60 
    ) 

])

In [ ]:
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

original_files = [
    f for f in os.listdir(INPUT_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
]

num_originals = len(original_files)

if num_originals == 0:
    raise ValueError("No images found in input directory.")

augment_needed = MAX_CLASS_REFERENCE - num_originals

print(f"Original images: {num_originals}")
print(f"Target images: {MAX_CLASS_REFERENCE}")
print(f"Images to generate: {max(augment_needed, 0)}")

In [ ]:
if augment_needed <= 0:

    quota_list = [0] * num_originals

else:

    base_augments = augment_needed // num_originals
    remainder = augment_needed % num_originals

    quota_list = (
        [base_augments + 1] * remainder
        + [base_augments] * (num_originals - remainder)
    )

    random.shuffle(quota_list)

print(f"Base augment per image: {augment_needed // num_originals if augment_needed > 0 else 0}")
print(f"Extra augmentation: {remainder if augment_needed > 0 else 0}")

In [ ]:
generated = 0

for filename, quota in zip(original_files, quota_list):

    if quota == 0:
        continue

    image_path = os.path.join(
        INPUT_IMAGE_DIR,
        filename
    )

    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        print(f"Cannot read: {filename}")
        continue

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    base, ext = os.path.splitext(filename)

    for _ in range(quota):

        augmented = augment_pipeline(
            image=image_rgb
        )["image"]

        output_name = (
            f"{base}_aug_{generated}{ext}"
        )

        output_path = os.path.join(
            OUTPUT_IMAGE_DIR,
            output_name
        )

        cv2.imwrite(
            output_path,
            cv2.cvtColor(
                augmented,
                cv2.COLOR_RGB2BGR
            )
        )

        generated += 1

print(f"Generated images: {generated}")

In [ ]:
for filename in original_files:

    source = os.path.join(
        INPUT_IMAGE_DIR,
        filename
    )

    destination = os.path.join(
        OUTPUT_IMAGE_DIR,
        filename
    )

    if not os.path.exists(destination):
        shutil.copy2(
            source,
            destination
        )

In [ ]:
final_files = [
    f for f in os.listdir(OUTPUT_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
]

print(f"Original images: {num_originals}")
print(f"Generated images: {generated}")
print(f"Final images: {len(final_files)}")
print(f"Target images: {MAX_CLASS_REFERENCE}")